In [ ]:
from _init import *

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

In [ ]:
import random, re, torch, seaborn, math
from transformers import PreTrainedTokenizerFast, AutoModelForCausalLM

from msnap.utils import common_const, common_utils, json_utils, container_utils, file_utils, model_utils, tokenizer_utils
from msnap.utils.container_utils import chunks
from msnap.utils.model_utils import is_correct

import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [ ]:
SEED = 42
common_utils.set_seed(SEED)

In [ ]:
work_dir = f'/home/nlpshlee/dev_env/git/repos/msnap'
data_dir = f'{work_dir}/data'

# zero_shot_target = 'fact'
zero_shot_target = 'other'

in_dir_target = f'{data_dir}/check_targets/zero_shot_{zero_shot_target}'
in_dir_hidden = f'{data_dir}/save_hiddens/zero_shot_{zero_shot_target}'
out_dir = f'{data_dir}/evaluate_msnap/zero_shot_{zero_shot_target}'

In [ ]:
model_name = 'Llama-3.2-3B'
# model_name = 'Llama-3.1-8B'

model_name_or_path = f'meta-llama/{model_name}-Instruct'
dtype = 'bfloat16'
device = 'cuda:0'
max_seq_length = 4096
max_new_tokens = 64

model: AutoModelForCausalLM = model_utils.get_model(model_name_or_path, dtype, device, is_eval=True)

# 평가/추론 시에는 반드시 'left' 패딩
tokenizer: PreTrainedTokenizerFast = tokenizer_utils.load_tokenizer(model_name_or_path, 'left')

num_layers = model.config.num_hidden_layers
print(f'\nevaluate_msnap() num_layers : {num_layers}')

In [ ]:
def get_all_layer_misinfo_vecs(in_prefix, n_components=10, pc_idx=0):
    max_mean_shift = common_const.NULL_FLOAT
    max_layer_idx = -1
    misinfo_vecs_dict = {}

    for layer_idx in range(1, num_layers+1):
        # 1. 데이터 로드
        fact_file_path = f'{in_prefix}/fact/layer_{layer_idx}_h_states.pt'
        counter_file_path = f'{in_prefix}/counter/layer_{layer_idx}_h_states.pt'

        if (not file_utils.exists(fact_file_path)) or (not file_utils.exists(counter_file_path)):
            continue

        layer_h_states_fact = torch.load(fact_file_path, weights_only=True).numpy()
        layer_h_states_counter = torch.load(counter_file_path, weights_only=True).numpy()

        # 2. 차이 벡터 계산
        delta_hs = layer_h_states_counter - layer_h_states_fact

        # 3. 데이터 스케일링 (PCA 전 필수 과정)
        scaler = StandardScaler()
        delta_scaled = scaler.fit_transform(delta_hs)

        # 4. PCA 수행
        pca = PCA(n_components=n_components)
        delta_pca = pca.fit_transform(delta_scaled)

        # 5. 히든 차원과 동일한 크기의 기저 벡터(화살표)
        V = pca.components_

        print(f'get_misinfo_vecs() [Layer {layer_idx}] delta_hs : {delta_hs.shape}, delta_pca : {delta_pca.shape}, V : {V.shape}')

        # 6. Mean Shift 계산
        v_pc = V[pc_idx]
        shifts = np.dot(delta_hs, v_pc)
        mean_shift = np.mean(shifts)

        if mean_shift < 0:
            V[pc_idx] = -V[pc_idx]
            mean_shift = -mean_shift
        
        # 7. 최대 확인
        if max_mean_shift < mean_shift:
            max_mean_shift = mean_shift
            max_layer_idx = layer_idx
        
        # 8. 모든 레이어 저장
        misinfo_vecs_dict[layer_idx] = V.copy()
    
    print(f'\nget_misinfo_vecs() [MAX] layer {max_layer_idx}, mean shift : {max_mean_shift}\n')

    return max_layer_idx, misinfo_vecs_dict

In [ ]:
def visualize_transition_matrix(cnt_f2f, cnt_f2c, cnt_f2o,
                                cnt_c2f, cnt_c2c, cnt_c2o,
                                cnt_o2f, cnt_o2c, cnt_o2o):

    # 1. 3x3 카운트 행렬 생성
    counts = np.array([
        [cnt_f2f, cnt_f2c, cnt_f2o],
        [cnt_c2f, cnt_c2c, cnt_c2o],
        [cnt_o2f, cnt_o2c, cnt_o2o]
    ])

    print(f'visualize_transition_matrix() counts :\n{counts}\n')

    # 2. 행 단위(Row-wise) 퍼센트 계산
    row_sums = counts.sum(axis=1)[:, np.newaxis]
    row_sums = np.where(row_sums == 0, 1, row_sums)
    percentages = (counts / row_sums) * 100

    # 3. 셀에 출력할 텍스트
    annot_data = np.empty_like(counts, dtype=object)
    for i in range(3):
        for j in range(3):
            annot_data[i, j] = f"{counts[i, j]}\n({percentages[i, j]:.2f}%)"
    
    # 4. 시각화
    plt.figure(figsize=(8, 6))
    labels = ['Fact', 'Counter', 'Other']

    ax = seaborn.heatmap(counts, annot=annot_data, fmt='', cmap='Blues', xticklabels=labels, yticklabels=labels,
                         cbar_kws={'label': 'Number of Prompts'}, annot_kws={"size": 12, "weight": "bold"})
    
    plt.title('State Transition Matrix (Original vs. Post Intervention)', fontsize=15, pad=15, fontweight='bold')
    plt.xlabel('Post Intervention (MSNAP)', fontsize=13, fontweight='bold', labelpad=10)
    plt.ylabel('Original (No Intervention)', fontsize=13, fontweight='bold', labelpad=10)

    # 행/열의 의미를 명확히 하기 위해 축 라벨 위치 조정
    ax.xaxis.tick_top()
    ax.xaxis.set_label_position('top')

    plt.tight_layout()
    plt.show()

    
    plt.close()

In [ ]:
def evaluate_msnap(datas: list, target_layer_idxs, all_layer_misinfo_vecs, alpha, batch_size):
    cnt_f2f, cnt_f2c, cnt_f2o = 0, 0, 0
    cnt_c2f, cnt_c2c, cnt_c2o = 0, 0, 0
    cnt_o2f, cnt_o2c, cnt_o2o = 0, 0, 0

    print(f'evaluate_msnap() data size {len(datas)}')
    for batch_idx, datas_batch in enumerate(chunks(datas, batch_size)):
        prompts = [data['prompt'] for data in datas_batch]
        answers_fact = [data['answer_fact'] for data in datas_batch]
        answers_counter = [data['answer_counter'] for data in datas_batch]

        generated_texts_org = model_utils.get_generated_texts(model, tokenizer, device, prompts, max_seq_length, max_new_tokens)

        generated_texts_msnap = model_utils.get_generated_texts(
            model, tokenizer, device, prompts, max_seq_length, max_new_tokens,
            target_layer_idxs=target_layer_idxs,
            all_layer_misinfo_vecs=all_layer_misinfo_vecs,
            alpha=alpha
        )

        for text_org, text_msnap, answer_fact, answer_counter in zip(generated_texts_org, generated_texts_msnap, answers_fact, answers_counter):
            if is_correct(text_org, answer_fact)[1]:
                if is_correct(text_msnap, answer_fact)[1]:
                    cnt_f2f += 1
                elif is_correct(text_msnap, answer_counter)[1]:
                    cnt_f2c += 1
                else:
                    cnt_f2o += 1
            elif is_correct(text_org, answer_counter)[1]:
                if is_correct(text_msnap, answer_fact)[1]:
                    cnt_c2f += 1
                elif is_correct(text_msnap, answer_counter)[1]:
                    cnt_c2c += 1
                else:
                    cnt_c2o += 1
            else:
                if is_correct(text_msnap, answer_fact)[1]:
                    cnt_o2f += 1
                elif is_correct(text_msnap, answer_counter)[1]:
                    cnt_o2c += 1
                else:
                    cnt_o2o += 1
        
        if (batch_idx+1) % 100 == 0:
            print(f'evaluate_msnap() {(batch_idx+1)*batch_size} complet.')
    print(f'evaluate_msnap() {len(datas)} complet.\n')
    
    visualize_transition_matrix(
        cnt_f2f, cnt_f2c, cnt_f2o,
        cnt_c2f, cnt_c2c, cnt_c2o,
        cnt_o2f, cnt_o2c, cnt_o2o
    )

In [ ]:
def get_batch_size(model_name, extract_fact_size, extract_counter_size):
    batch_size = 20

    if '8B' in model_name:
        if (extract_fact_size + extract_counter_size) <= 5:
            batch_size = 5
        elif (extract_fact_size + extract_counter_size) <= 10:
            batch_size = 2
        else:
            batch_size = 1
    
    return batch_size

In [ ]:
def get_window(target_idx, n_window):
    # 현재 0번째는 임베딩 벡터이고, 레이어 인덱스는 1부터라서, 모델에게 실제 인덱스를 넘겨줄 때는 '-1'
    target_idx -= 1

    if n_window < 1:
        return [target_idx]

    min_idx = max(0, target_idx - n_window)
    max_idx = min(target_idx + n_window, num_layers-1)

    idxs = [idx for idx in range(min_idx, target_idx)]
    idxs += [idx for idx in range(target_idx, max_idx+1)]

    return idxs

In [ ]:
def get_sliced(target_layer_idxs: list, all_layer_misinfo_vecs: dict, n_slice: int):
    sliced = {}

    # target_layer_idx는 get_window()에서 0부터 시작
    for target_layer_idx in target_layer_idxs:
        # 현재 0번째는 임베딩 벡터이고, 레이어 인덱스는 1부터라서, '0'번째 레이어는 저장된 취약성 벡터에서 '1'번째
        sliced[target_layer_idx] = all_layer_misinfo_vecs[target_layer_idx+1][0:n_slice, :]
    
    return sliced

In [ ]:
# extract_size_pairs = []

# for extract_fact_size in range(1, 11):
#     for extract_counter_size in range(1, 11):
#         extract_size_pairs.append([extract_fact_size, extract_counter_size])

extract_size_pairs = [[2,8], [8,2], [3,7], [7,3], [1,3], [3,1]]

context_orders = ['shuffle']

In [ ]:
for extract_size_pair in extract_size_pairs:
    extract_fact_size, extract_counter_size = extract_size_pair[0], extract_size_pair[1]
    batch_size = get_batch_size(model_name, extract_fact_size, extract_counter_size)

    for context_order in context_orders:
        print(f'extract_fact_size : {extract_fact_size}, extract_counter_size : {extract_counter_size}, context_order : {context_order}\n')

        # 1. 저장된 히든 벡터에서 '오정보 취약 기저' 추출
        in_prefix = f'{in_dir_hidden}/{model_name}/{context_order}/F-{extract_fact_size}_C-{extract_counter_size}'
        max_layer_idx, all_layer_misinfo_vecs = get_all_layer_misinfo_vecs(in_prefix)

        # 2. 평가를 위한 변수 설정 및 타겟 데이터 로드
        windows = [0, 1, 2, 3]
        alphas = [1, 2.5, 5, 10]

        in_prefix = f'{in_dir_target}/{model_name}/checked/{context_order}/F-{extract_fact_size}'
        datas_list = []

        in_file_path = f'{in_prefix}/msnap_gpt-5.2_checked_targets_{model_name}_{context_order}_F-{extract_fact_size}_C-{extract_counter_size}_A-C.json'
        datas_counter = json_utils.load_json(in_file_path)
        datas_list.append(datas_counter)

        in_file_path = f'{in_prefix}/msnap_gpt-5.2_checked_targets_{model_name}_{context_order}_F-{extract_fact_size}_C-{extract_counter_size}_A-F.json'
        datas_fact = json_utils.load_json(in_file_path)
        datas_list.append(datas_fact)

        in_file_path = f'{in_prefix}/msnap_gpt-5.2_checked_targets_{model_name}_{context_order}_F-{extract_fact_size}_C-{extract_counter_size}_A-O.json'
        datas_other = json_utils.load_json(in_file_path)
        datas_list.append(datas_other)

        for datas, data_type in zip(datas_list, ['A-C', 'A-F', 'A-O']):
            for window in windows:
                for alpha in alphas:
                    target_layer_idxs = get_window(max_layer_idx, window)

                    for n_slice in range(1, 4):
                        print(f'\n# [{data_type}] window={window}, target_layer_idxs={target_layer_idxs}, alpha={alpha}, n_slice={n_slice}\n')
                        evaluate_msnap(datas, target_layer_idxs, get_sliced(target_layer_idxs, all_layer_misinfo_vecs, n_slice), alpha, batch_size)